# Phân loại bệnh tim: Decision Tree, Random Forest và XGBoost

Notebook chạy trên Google Colab: upload `heart.csv` ở cell đầu tiên, sau đó chạy lần lượt các cell bên dưới.

In [ ]:
# IMPORT FILE DỮ LIỆU VÀO ĐÂY: chọn file heart.csv từ máy tính
from google.colab import files
uploaded = files.upload()
DATA_PATH = next(iter(uploaded))
print('Đã upload:', DATA_PATH)

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [ ]:
df = pd.read_csv(DATA_PATH)
print('Kích thước dữ liệu:', df.shape)
df.head()

In [ ]:
df.info()
print('\nSố giá trị thiếu mỗi cột:')
display(df.isna().sum())
print('\nTỷ lệ nhãn HeartDisease:')
display(df['HeartDisease'].value_counts(normalize=True).rename('Tỷ lệ'))

In [ ]:
# Tách X/y và chia dữ liệu: 80% train, 20% test
X = df.drop(columns='HeartDisease')
y = df['HeartDisease']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

categorical_cols = X.select_dtypes(include=['object', 'string']).columns
numeric_cols = X.select_dtypes(exclude=['object', 'string']).columns

# One-hot encoder chỉ được fit trên train để tránh data leakage
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ('num', 'passthrough', numeric_cols)
])
X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

print('Train:', X_train_ready.shape, '| Test:', X_test_ready.shape)
results = {}

In [ ]:
def evaluate_model(name, model):
    start = time.perf_counter()
    model.fit(X_train_ready, y_train)
    train_time = time.perf_counter() - start

    train_acc = accuracy_score(y_train, model.predict(X_train_ready))
    test_acc = accuracy_score(y_test, model.predict(X_test_ready))
    results[name] = {
        'Train accuracy': train_acc,
        'Test accuracy': test_acc,
        'Chênh lệch train-test': train_acc - test_acc,
        'Thời gian train (giây)': train_time
    }
    print(f'Train accuracy: {train_acc:.3f}')
    print(f'Test accuracy:  {test_acc:.3f}')
    print(f'Chênh lệch:     {train_acc - test_acc:.3f}')
    print(f'Thời gian train: {train_time:.3f} giây')
    return model

def plot_importance(model, title):
    importance = pd.Series(model.feature_importances_, index=feature_names)
    importance = importance.sort_values().tail(12)
    ax = importance.plot.barh(figsize=(8, 5), color='steelblue')
    ax.set_title(title)
    ax.set_xlabel('Feature importance')
    plt.show()
    print('Feature quan trọng nhất:', importance.index[-1])

## 1. Decision Tree

In [ ]:
decision_tree = DecisionTreeClassifier(random_state=42)
decision_tree = evaluate_model('Decision Tree', decision_tree)
plot_importance(decision_tree, 'Decision Tree - Feature importance')

## 2. Random Forest

In [ ]:
random_forest = RandomForestClassifier(
    n_estimators=200, random_state=42, n_jobs=-1
)
random_forest = evaluate_model('Random Forest', random_forest)
plot_importance(random_forest, 'Random Forest - Feature importance')

## 3. XGBoost

In [ ]:
xgboost = XGBClassifier(
    n_estimators=200, max_depth=3, learning_rate=0.05,
    random_state=42, n_jobs=-1, eval_metric='logloss'
)
xgboost = evaluate_model('XGBoost', xgboost)
plot_importance(xgboost, 'XGBoost - Feature importance')

## Bảng tổng kết và nhận xét

In [ ]:
summary = pd.DataFrame(results).T
display(summary.style.format({
    'Train accuracy': '{:.3f}',
    'Test accuracy': '{:.3f}',
    'Chênh lệch train-test': '{:.3f}',
    'Thời gian train (giây)': '{:.3f}'
}))

models = {
    'Decision Tree': decision_tree,
    'Random Forest': random_forest,
    'XGBoost': xgboost
}
top_features = {
    name: feature_names[np.argmax(model.feature_importances_)]
    for name, model in models.items()
}
display(pd.Series(top_features, name='Feature quan trọng nhất'))

print('Nhận xét: test accuracy cao hơn nghĩa là dự đoán tốt hơn trên tập test.')
print('Chênh lệch train-test càng nhỏ thì mô hình càng ít có dấu hiệu overfitting.')
print('Nếu ba feature trên giống hoặc gần giống nhau, tầm quan trọng là nhất quán giữa các model.')